# FastAPI – Practice Exercises

Hands-on exercises for FastAPI, following `g_fw-fastapi.ipynb`.

Use `from fastapi.testclient import TestClient` to test without a running server.

### Contents

- [Exercise 1 – Hello World App](#exercise-1--hello-world-app)
- [Exercise 2 – Path, Query, and Body Parameters](#exercise-2--path-query-and-body-parameters)
- [Exercise 3 – Pydantic Request and Response Models](#exercise-3--pydantic-request-and-response-models)
- [Exercise 4 – Dependency Injection](#exercise-4--dependency-injection)
- [Exercise 5 – Background Tasks](#exercise-5--background-tasks)
- [Exercise 6 – Error Handling and Custom Responses](#exercise-6--error-handling-and-custom-responses)
- [Exercise 7 – Testing with TestClient](#exercise-7--testing-with-testclient)
- [Exercise 8 – Async Routes and External Services](#exercise-8--async-routes-and-external-services)
- [Exercise 9 – Dependency Overrides for Testing](#exercise-9--dependency-overrides-for-testing)
- [Exercise 10 – Project Structure and Best Practices](#exercise-10--project-structure-and-best-practices)

In [27]:
import os, sys, numpy, asyncio, aiohttp, requests, re
from typing import Any, List, Tuple, Dict, Set, Callable
from pandas import Series, DataFrame, Index, concat
from pandas import Timestamp, Timedelta, date_range
from enum import Enum
from dataclasses import dataclass

from fastapi import FastAPI, BackgroundTasks
from fastapi import Query, Body, Depends
from fastapi.testclient import TestClient
from fastapi.routing import APIRoute
from pydantic import BaseModel, Field

## Support classes

In [28]:
SYMBOLS = Index(filter(len, re.sub("(\n| )+", " ", """
    AAVE ADA AGIX APT ARB ATOM AVAX AXS BNB BTC CAKE CHZ COMP CRV DOGE DOT DYDX ENJ
    ETC ETH FET FIL FLOW GALA ICP IMX INJ LDO LINK LTC MANA MATIC MKR NEAR OCEAN OP
    PEPE RNDR SAND SEI SHIB SNX SOL STX SUI SUSHI TIA UNI XRP YFI
""").split(" ")))

In [29]:
@dataclass(frozen = True) ## SYMBOL
class Symbol:
    venue: str; symbol: str; digits: int = 5
    point: float = 0.00001; min_qty: float = 0.01
    bod: Timedelta = None; eod: Timedelta = None
    TEST_SYMBOLS = {"BTCUSDT": 2, "ETHUSDT": 3, "BNBUSDT": 4}
    @classmethod
    def sample(cls):
        test_symbols = list(cls.TEST_SYMBOLS.keys())
        test_symbol = str(numpy.random.choice(test_symbols))
        digits = cls.TEST_SYMBOLS[test_symbol]
        point = round(pow(0.1, digits), 8)
        return cls("BINANCE", test_symbol, digits, point)
    @property
    def tradable(self):
        now = Timestamp.utcnow()
        dnow = now - now.floor("1D")
        return (None in [self.bod, self.bod]) \
            or (self.bod <= dnow < self.eod) \
            or (dnow < self.eod <= self.bod) \
            or (self.eod <= self.bod < dnow)

In [30]:
@dataclass(frozen = True) ## TICK
class Tick:
    pa: float; pb: float; va: float; vb: float
    venue: str; symbol: str; time: Timestamp = None
    def __post_init__(self):
        if self.time is None:
            self.time = Timestamp.utcnow()
    @property
    def str_data(self):
        return f"A{self.va:.2f}@{self.pa:.5f} B{self.vb:.2f}@{self.pb:.5f}"
    def __repr__(self):
        time = self.time.isoformat(" ", "microseconds")[: 26]
        return f"Tick({time} :: {self.venue}|{self.symbol} :: {self.str_data})"

In [31]:
@dataclass(frozen = True) ## OHLC
class OHLC:
    timestamp: Timestamp; timeframe: Timedelta
    open: float; high: float; low: float; close: float
    @classmethod
    def from_series(cls, ts: Series, tf: Timedelta = None):
        pl, ph = sys.float_info.max, sys.float_info.min
        to = Timestamp.max.tz_localize("UTC")
        tc = Timestamp.min.tz_localize("UTC")
        for time, value in ts.items():
            if (tc < time): pc, tc = value, time
            if (time < to): po, to = value, time
            if (ph < value): ph = value
            if (pl > value): pl = value
        if (tf is None): tf = tc - to
        else: to = to.floor(tf)
        return cls(to, tf, float(po), float(ph), float(pl), float(pc))

In [32]:
@dataclass(frozen = True) ## CANDLE
class Candle:
    symbol: Symbol; ask: OHLC; bid: OHLC
    vavg: float; vwap: float; tvol: int
    def __post_init__(self):
        now = Timestamp.utcnow()
        object.__setattr__(self, "time", now)
    @classmethod
    def from_ticks(cls, df: DataFrame, symbol: Symbol = None, tf: Timedelta = None):
        if symbol is None: symbol = Symbol.sample()
        is_ask = df.columns.str.contains("a", regex = True)
        is_bid = df.columns.str.contains("b", regex = True)
        is_price = df.columns.str.contains("p", regex = True)
        is_vol = df.columns.str.contains("q|v", regex = True)
        column_ap: str = df.columns[is_ask & is_price][0]
        column_bp: str = df.columns[is_bid & is_price][0]
        column_av: str = df.columns[is_ask & is_vol][0]
        column_bv: str = df.columns[is_bid & is_vol][0]
        df_ap, df_av = df[column_ap], df[column_av].diff().abs()
        df_bp, df_bv = df[column_bp], df[column_bv].diff().abs()
        ask, bid = OHLC.from_series(df_ap, tf), OHLC.from_series(df_bp, tf)
        vwap = Series.mean((df_ap * df_av + df_bp * df_bv) / (df_av + df_bv))
        vwap = float(round(vwap, symbol.digits))
        vavg = int(Series.mean(df_av + df_bv))
        return cls(symbol, ask, bid, vavg, vwap, df.shape[0])
    @classmethod
    def sample(cls, size: int = 12345, pgen = 1.23456,
               tf: Timedelta = Timedelta(minutes = 1)):
        tgen = Timestamp.utcnow().floor(tf)
        index = date_range(tgen, tgen + tf, size + 1)
        df = DataFrame(index = index[: -1])
        columns = ["ap", "bp", "av", "bv"]
        df[columns] = numpy.random.randn(size, 4)
        delta = pow(10, int(numpy.log10(pgen / size) - 2))
        df["bp"] = numpy.sign(df["bp"].shift()).cumsum()
        df["ap"] = (df["ap"] * 5).abs().round()
        df["bp"] = delta * df["bp"].fillna(0) + pgen
        df["ap"] = delta * (df["ap"] + 1) + df["bp"]
        df["av"] = ((df["av"] * 50).abs() + size).astype(int)
        df["bv"] = ((df["bv"] * 50).abs() + size).astype(int)
        return cls.from_ticks(df, tf = tf)

In [33]:
@dataclass(frozen = True) ## ORDER
class Order:
    quantity: float
    symbol: Symbol
    price: float = None
    comment: str = None
    def __post_init__(self):
        if abs(self.quantity) < (min_qty := self.symbol.min_qty):
            error = f"\"{abs(self.quantity):.2f} < {min_qty}\""
            raise ValueError(f"Quantity below min: " + error)
        if not self.comment: object.__setattr__(self, "comment", "")
        _type: Order.Type = None if self.price else Order.Type.MARKET
        _side: Order.Side = Order.Side(int(numpy.sign(self.quantity)))
        quantity = round(abs(self.quantity) / min_qty) * min_qty
        object.__setattr__(self, "quantity", quantity)
        object.__setattr__(self, "_type", _type)
        object.__setattr__(self, "_side", _side)
        t = int(Timestamp.utcnow().timestamp() * 1e6)
        id = numpy.base_repr(t, 36)
        object.__setattr__(self, "ts", t)
        object.__setattr__(self, "id", id)

    class Side(Enum): BUY, SELL = +1, -1
    class Type(Enum): MARKET, LIMIT, STOP = 0, -1, +1
    class Status(Enum): PLACED, PENDING, CANCELLED, REJECTED = 0, 1, 2, 3
    @property
    def __dict__(self): return {"ts": self.ts, "id": self.id,
        "venue": self.symbol.venue, "symbol": self.symbol.symbol,
        "quantity": abs(self.quantity), "price": self.price,
        "type": self.type.name, "side": self.side.name,
        "comment": self.comment}
    @property
    def type(self): return self._type
    @property
    def side(self): return self._side
    def on_price(self, price: float):
        assert isinstance(price, float) and (price > 0)
        _side: Order.Side = object.__getattribute__(self, "_side")
        _type: Order.Type = object.__getattribute__(self, "_type")
        if _type: object.__setattr__(self, "price", price)
        dist = (self.price - price) * _side.value
        _type = Order.Type(numpy.sign(dist))
        object.__setattr__(self, "_type", _type)

In [34]:
class Position(BaseModel): ## POSITION
    ots: int = Field(..., description = "Order timestamp in microseconds")
    oid: str = Field(..., description = "Order UID, equal to timestamp in base-36")
    pts: int = Field(..., description = "Position timestamp in microseconds")
    pid: str = Field(..., description = "Position UID, equal to timestamp in base-36")
    venue: str = Field(..., description = "Venue/exchange")
    symbol: str = Field(..., description = "Symbol for the instrument")
    quantity: float = Field(..., description = "Position quantity")
    price: float | None = Field(None, description = "Fill or order price")
    side: str = Field(..., description = "Order side")
    def __init__(self, **kwargs):
        ots, oid = kwargs.pop("ts"), kwargs.pop("id")
        pts = int(Timestamp.utcnow().timestamp() * 1e6)
        pid = numpy.base_repr(pts, 36)
        kwargs.update({"ots": ots, "pts": pts, "oid": oid, "pid": pid})
        super().__init__(**kwargs)

    @classmethod
    def from_order(cls, order: Order): return cls(**order.__dict__)

In [35]:
class Binance: ## BINANCE/HTTP

    BASE_URL = "https://api.binance.com/api/v3"

    @classmethod
    def log(cls, *args):
        t = Timestamp.utcnow()
        ts = t.strftime("%H:%M:%S.%f")
        print(f"[{ts[: -3]}]", *args)
        return t

    @classmethod
    async def _request(cls, session: aiohttp.ClientSession,
                semaphore: asyncio.Semaphore, args: dict):

        async with semaphore:
            async with session.get(**args) as response:
                try: response.raise_for_status()
                except Exception as EXC:
                    cls.log(f"Response error: \"{EXC!r}\"")
                    return None
                try: data = await response.json()
                except Exception as EXC:
                    cls.log(f"Resp data error: \"{EXC!r}\"")
                    return None
                return data

    @classmethod
    async def price(cls, symbols: List[str], timeout: float = 2, max_threads: int = 4):


        args = dict[str, Any](
            url = f"{cls.BASE_URL}/ticker/price",
            timeout = aiohttp.ClientTimeout(timeout))
        
        semaphore = asyncio.Semaphore(max_threads)
        if isinstance(symbols, str): symbols = [symbols]
        async with aiohttp.ClientSession() as session:
            tasks = dict.fromkeys(symbols)
            for symbol in symbols:
                args["params"] = {"symbol": symbol}
                future = cls._request(session, semaphore, args.copy())
                tasks[symbol] = asyncio.create_task(future)
            results = dict(zip(
                symbols, await asyncio.gather(*tasks.values())))
            return {K: float(V["price"]) for K, V in results.items()}

## Exercise 1 – Hello World App

**Goal**: Create a minimal FastAPI application.

**Requirements**:

- Instantiate `app = FastAPI()`.
- Add a `GET /` route that returns `{"message": "hello"}`.
- Use `TestClient(app)` to call the route and assert the response.

**Hint**: `TestClient` wraps `requests` and runs the app in-process.

In [36]:
import asyncio, aiohttp
app = FastAPI(title = "EXERCISE 1")

@app.get("/test")
async def root():
    return {"message": "Hello!"}

with TestClient(app) as client:
    response = client.get("test")
    print(response.json())

{'message': 'Hello!'}


## Exercise 2 – Path, Query, and Body Parameters

**Goal**: Accept different kinds of input in a single endpoint.

**Requirements**:

- Add `GET /symbols/{ticker}` that returns the ticker as JSON.
- Add a `limit: int = 10` query parameter.
- Add `POST /orders` that accepts a JSON body with `symbol`, `qty`, `price`.

**Hint**: Path params are declared as function arguments matching the `{name}` in the route.

In [37]:
class API2:

    _APP = FastAPI(title = "EXERCISE 2")
    _DESC = dict(
        price = "List of symbol tickers"
    )

    @_APP.get("/price")
    async def price(symbols: list = Query(
        None, description = _DESC["price"])):
        return await Binance.price(symbols, timeout = 60)

    @_APP.post("/order")
    async def order(body: dict = Body()):
        symbol = body.pop("symbol")
        resp = await Binance.price(symbol)
        symbol = Symbol("BINANCE", symbol)
        order = Order(symbol = symbol, **body)
        order.on_price(resp[symbol.symbol])
        return order.__dict__

with TestClient(API2._APP) as client:
    
    symbol = Symbol.sample()
    params = {"symbols": list(Symbol.TEST_SYMBOLS)}
    response = client.get("price", params = params).json()
    print("Prices:", response)
    price = response[symbol.symbol]
    sign = numpy.sign(numpy.random.random() - 0.5)
    quantity = sign / (100 * symbol.point)
    price = price - price * 0.01 * sign
    response = client.post("order", json = {
        "symbol": symbol.symbol, "price": price,
        "quantity": quantity}).json()
    print("Order:", response)

Prices: {'BTCUSDT': 68576.87, 'ETHUSDT': 2105.2, 'BNBUSDT': 598.93}
Order: {'ts': 1775544847954029, 'id': 'HHDM1J9G19', 'venue': 'BINANCE', 'symbol': 'ETHUSDT', 'quantity': 10.0, 'price': 2084.1479999999997, 'type': 'LIMIT', 'side': 'BUY', 'comment': ''}


## Exercise 3 – Pydantic Request and Response Models

**Goal**: Validate and document request/response shapes.

**Requirements**:

- Define `OrderIn(BaseModel)` with `symbol: str`, `qty: int`, `price: float`.
- Define `OrderOut(OrderIn)` adding `id: int` and `created_at: datetime`.
- Use `response_model=OrderOut` on the POST route; return a fake `OrderOut` object.

**Hint**: `response_model` filters the output to only the declared fields.

In [38]:
class API3(API2):

    _APP = API2._APP

    @_APP.post("/position", response_model = Position)
    async def position(body: dict = Body()):
        return Position(**body)

with TestClient(API3._APP) as client:

    symbol = Symbol.sample()
    params = {"symbols": list(Symbol.TEST_SYMBOLS)}
    response = client.get("price", params = params).json()
    price = response[symbol.symbol]
    sign = numpy.sign(numpy.random.random() - 0.5)
    quantity = sign / (100 * symbol.point)
    price = price - price * 0.01 * sign
    response = client.post("order", json = {
        "symbol": symbol.symbol, "price": price,
        "quantity": quantity}).json()
    position = client.post("position", json = response)
    print("Position status:", position)
    print("Position content:", position.json())

Position status: <Response [200 OK]>
Position content: {'ots': 1775544848710370, 'oid': 'HHDM1JPNMQ', 'pts': 1775544848711738, 'pid': 'HHDM1JPOOQ', 'venue': 'BINANCE', 'symbol': 'BNBUSDT', 'quantity': 100.0, 'price': 604.9192999999999, 'side': 'SELL'}


## Exercise 4 – Dependency Injection

**Goal**: Share reusable dependencies across routes.

**Requirements**:

- Write a `get_db()` generator that yields a fake in-memory dict.
- Inject it into a route with `db: dict = Depends(get_db)`.
- Write a `get_current_user()` dependency that reads a `token` query param and raises `HTTPException(401)` if missing.

**Hint**: `Depends(fn)` calls `fn` for each request and injects the returned value.

In [39]:
prices = await Binance.price(SYMBOLS + "USDT", timeout = 60, max_threads = 8)

class DB(DataFrame):

    def __init__(self, n: int = 500):
        db = Series()
        shuffled = numpy.random.choice([*prices], n)
        shuffled = list(map(str, shuffled))
        for n, symbol in enumerate(shuffled):
            price = prices.get(symbol, None)
            if (price is None): continue
            comment = f"sample_{n:04.0f}"
            digits = round(numpy.log10(price))
            point = round(pow(0.1, 5 - digits), 12)
            min_qty = round(1 / (100 * point), 2)
            symbol = Symbol("BINANCE", str(symbol), digits, point, min_qty)
            quantity = int(1 + numpy.random.random() * 9) * min_qty
            rand_side = int(numpy.sign(numpy.random.random() * 2 - 1) * 1)
            rand_type = int(numpy.sign(numpy.random.random() * 2 - 1) * 20)
            params = {"quantity": rand_side * quantity, "symbol": symbol,
                  "price": price + rand_type * point, "comment": comment}
            order = Order(**params)
            order.on_price(price)
            db[order.id] = order.__dict__

        db = db.apply(Series)
        db = db.set_index("id").sort_index()
        db["price_limit"] = db.pop("price")
        db["price_last"] = db["symbol"].map(prices)
        db["type"] = db["type"].map(Order.Type._member_map_)
        db["side"] = db["side"].map(Order.Side._member_map_)
        db["notional"] = db["side"].map(lambda x: x.value)
        db["notional"] *= db["quantity"] * db["price_limit"]
        super().__init__(db)

    def db(self): return self

In [40]:
class API4:
    _APP = FastAPI(title = "EXERCISE 4")
    _DB = DB(600)

    @_APP.get("/orders/symbols")
    async def get_orders_by_symbol(
        symbols: List[str] = Query(..., description = "List of symbols"),
        db: DataFrame = Depends(_DB.db)):

        df_response: DataFrame = db.loc[db["symbol"].isin(symbols)]
        items = df_response.to_dict(orient = "index")
        return {"response": "success", "items": items}

    @_APP.get("/orders/time")
    async def get_orders_by_time(
        since: str = Query(..., description = "Lower time of the range"),
        until: str = Query(None, description = "Upper time of the range"),
        db: DataFrame = Depends(_DB.db)):

        if until is None: until = Timestamp.utcnow()
        since_ts = int(Timestamp(since).timestamp() * 1e6)
        until_ts = int(Timestamp(until).timestamp() * 1e6)
        df_response: DataFrame = db.loc[db["ts"].between(since_ts, until_ts)]
        items = df_response.to_dict(orient = "index")
        return {"response": "success", "items": items}

max_rows_view = 8
with TestClient(API4._APP) as client:

    symbols = numpy.random.choice([*prices], 3)
    params = {"symbols": sorted(map(str, symbols))}
    response = client.get("orders/symbols", params = params).json()
    df = DataFrame.from_dict(response["items"], "index")
    df["ts"] = df["ts"].apply(Timestamp, unit = "us")

    print("By-symbol... Request", params["symbols"], end = " ==> ")
    print("Response:", response["response"], f"({df.shape[0]} rows)\n")
    print(df.to_string(max_rows = max_rows_view), end = "\n\n")

    since: Timestamp = df["ts"].iloc[df.shape[0] // 2]
    params = {"since": since.isoformat(" ", "milliseconds")}
    response = client.get("orders/time", params = params).json()
    df = DataFrame.from_dict(response["items"], "index")
    df["ts"] = df["ts"].apply(Timestamp, unit = "us")

    print("By-time... Request since", params["since"], end = " ==> ")
    print("Response:", response["response"], f"({df.shape[0]} rows)\n")
    print(df.to_string(max_rows = max_rows_view))

By-symbol... Request ['AGIXUSDT', 'SEIUSDT', 'SUIUSDT'] ==> Response: success (27 rows)

                                   ts    venue    symbol  quantity  type  side      comment  price_limit  price_last  notional
HHDM1KYUQV 2026-04-07 06:54:10.819111  BINANCE   SEIUSDT   70000.0     1     1  sample_0024      0.05272      0.0527    3690.4
HHDM1KYZ98 2026-04-07 06:54:10.824956  BINANCE  AGIXUSDT    8000.0     1    -1  sample_0058      0.61390      0.6141   -4911.2
HHDM1KZ1C1 2026-04-07 06:54:10.827649  BINANCE  AGIXUSDT    7000.0    -1    -1  sample_0073      0.61430      0.6141   -4300.1
HHDM1KZ3AY 2026-04-07 06:54:10.830202  BINANCE  AGIXUSDT    1000.0    -1     1  sample_0090      0.61390      0.6141     613.9
...                               ...      ...       ...       ...   ...   ...          ...          ...         ...       ...
HHDM1L0FE3 2026-04-07 06:54:10.892523  BINANCE  AGIXUSDT    7000.0    -1    -1  sample_0455      0.61430      0.6141   -4300.1
HHDM1L0UFD 2026-04-07 

## Exercise 5 – Background Tasks

**Goal**: Run work after the response is sent.

**Requirements**:

- Add a `BackgroundTasks` parameter to a POST route.
- Call `background_tasks.add_task(send_alert, order_id)` where `send_alert` logs a message.
- Verify the route returns immediately and the task runs afterwards.

**Hint**: `BackgroundTasks` is built-in; for heavy work prefer a real task queue like Celery.

In [41]:
def remove_route(app: FastAPI, endp: str):
    routes_to_keep = list()
    for route in app.routes:
        if not isinstance(route, APIRoute): continue
        if route.path.endswith(endp): continue
        routes_to_keep.append(route)
    app.routes[:] = routes_to_keep
    
class API5(API3):

    _APP = API3._APP
    remove_route(_APP, "position")

    def oms(order: Dict): print("Enqueuing order in OMS:\n =>", order)

    @_APP.post("/position", response_model = Position)
    async def position(background_tasks: BackgroundTasks, body: dict = Body()):
        background_tasks.add_task(API5.oms, body)
        return Position(**body)

with TestClient(API5._APP) as client:
    
    symbol = Symbol.sample()
    params = {"symbols": list(Symbol.TEST_SYMBOLS)}
    response = client.get("price", params = params).json()
    price = response[symbol.symbol]
    sign = numpy.sign(numpy.random.random() - 0.5)
    quantity = sign / (100 * symbol.point)
    price = price - price * 0.01 * sign
    response = client.post("order", json = {
        "symbol": symbol.symbol, "price": price,
        "quantity": quantity}).json()
    position = client.post("position", json = response)
    print("Position status:", position)
    print("Position content:", position.json())

Enqueuing order in OMS:
 => {'ts': 1775544851715108, 'id': 'HHDM1LI23O', 'venue': 'BINANCE', 'symbol': 'BNBUSDT', 'quantity': 100.0, 'price': 592.9407, 'type': 'LIMIT', 'side': 'BUY', 'comment': ''}
Position status: <Response [200 OK]>
Position content: {'ots': 1775544851715108, 'oid': 'HHDM1LI23O', 'pts': 1775544851716905, 'pid': 'HHDM1LI3HL', 'venue': 'BINANCE', 'symbol': 'BNBUSDT', 'quantity': 100.0, 'price': 592.9407, 'side': 'BUY'}


## Exercise 6 – Error Handling and Custom Responses

**Goal**: Return meaningful error responses.

**Requirements**:

- Raise `HTTPException(status_code=404, detail='Symbol not found')` for an unknown ticker.
- Register an exception handler for `ValueError` that returns a 422 response with a message.
- Test both the 404 and 422 paths with `TestClient`.

**Hint**: `@app.exception_handler(ExcType)` registers a global handler for a given exception class.

In [42]:
from fastapi import HTTPException
class API6(API4):

    _APP = API4._APP
    _DB = API4._DB
    @_APP.get("/orders/id")
    async def get_orders_by_id(
        ids: List[str] = Query(...),
        db: DataFrame = Depends(_DB.db)):
        
        try: df_response: DataFrame = db.loc[ids]
        except:
            error_ids = sorted({*ids}.difference({*db.index}))
            error = "IDs nonexistent: " + str.join(", ", error_ids)
            raise HTTPException(404, error)
        items = df_response.to_dict(orient = "index")
        return {"response": "success", "items": items}

max_rows_view = 8
with TestClient(API6._APP) as client:

    symbols = numpy.random.choice([*prices], 3)
    params = {"symbols": sorted(map(str, symbols))}
    response = client.get("orders/symbols", params = params).json()
    df = DataFrame.from_dict(response["items"], "index")
    df["ts"] = df["ts"].apply(Timestamp, unit = "us")

    print("By-symbol... Request", params["symbols"], end = " ==> ")
    print("Response:", response["response"], f"({df.shape[0]} rows)\n")
    print(df.to_string(max_rows = max_rows_view), end = "\n\n")

    params = {"ids": ["ABC1", "ABC2", "ABC3", *df.index[-3 :]]}
    response = client.get("orders/id", params = params)

    print("By-symbol... Request", params["ids"])
    print("Response:", response, response.json())

By-symbol... Request ['APTUSDT', 'SEIUSDT', 'SHIBUSDT'] ==> Response: success (27 rows)

                                   ts    venue    symbol     quantity  type  side      comment  price_limit  price_last  notional
HHDM1KYTM5 2026-04-07 06:54:10.817645  BINANCE   APTUSDT       3000.0    -1    -1  sample_0013     0.834200    0.834000   -2502.6
HHDM1KYUQV 2026-04-07 06:54:10.819111  BINANCE   SEIUSDT      70000.0     1     1  sample_0024     0.052720    0.052700    3690.4
HHDM1KYWKF 2026-04-07 06:54:10.821471  BINANCE  SHIBUSDT  600000000.0     1    -1  sample_0039     0.000006    0.000006   -3508.8
HHDM1KYZ5H 2026-04-07 06:54:10.824821  BINANCE   APTUSDT       3000.0    -1    -1  sample_0057     0.834200    0.834000   -2502.6
...                               ...      ...       ...          ...   ...   ...          ...          ...         ...       ...
HHDM1L0OPW 2026-04-07 06:54:10.904612  BINANCE   APTUSDT       3000.0     1     1  sample_0520     0.834200    0.834000    2502.6
H

## Exercise 7 – Testing with TestClient

**Goal**: Write a full test suite for a small trading API.

**Requirements**:

- Build a `/orders` CRUD API (list, create, get by id) backed by an in-memory list.
- Write tests for: create order (201), get existing (200), get missing (404), list all (200).
- Assert response JSON fields and the length of the list after several creates.

**Hint**: Reset shared state between tests using a `setup_function` or dependency override.

In [43]:
from unittest import TestCase, TestLoader, TextTestRunner

class API7(API6):

    _APP = API6._APP
    _DB = DB(600)
    remove_route(_APP, "send")
    @_APP.post("/send")
    async def send(quantity: float = Query(...), symbol: str = Query(...),
        price: float = Query(None, description = "Order price (optional)"),
        comment: str = Query(None, description = "Order comment (optional)"),
        db: DataFrame = Depends(_DB.db)):

        price_last = (await Binance.price([symbol], 20))[symbol]
        digits = int(7 - round(numpy.log10(price_last)))
        point = round(pow(0.1, digits), 12)
        min_qty = round(1 / (1000000 * point), 12)
        symbol = Symbol("BINANCE", symbol, digits, point, min_qty)
        order = Order(quantity, symbol, price, comment)
        order.on_price(price_last)
        notional = order.quantity * order.price * order.side.value
        _side, _type = Order.Side(order.side), Order.Type(order.type)
        response = {"ts": order.ts, "venue": order.symbol.venue, "symbol": order.symbol.symbol,
            "quantity": order.quantity, "type": _type, "side": _side, "comment": order.comment,
            "price_limit": order.price, "price_last": price_last, "notional": notional}
        db.loc[order.id] = response
        response["id"] = order.id
        return response

class TestAPI(TestCase):

    KEYS = ["id", "ts", "quantity", "comment", "price_limit"]

    def setUp(self):
        self.client = TestClient(API7._APP)

    def test_send(self):
        orders = [
            {"quantity": +0.1, "symbol": "BTCUSDT"},
            {"quantity": -1.0, "symbol": "ETHUSDT"},
            {"quantity": +10.0, "symbol": "BNBUSDT"},
            {"quantity": -100.0, "symbol": "SOLUSDT"},
            {"quantity": +1000.0, "symbol": "UNIUSDT"},
            {"quantity": -10000.0, "symbol": "MATICUSDT"},
            {"quantity": +100000.0, "symbol": "FLOWUSDT"},
        ]
        for n, case in enumerate(orders):
            print("Testing", n, "... Case:",case)
            with self.subTest(case = case):
                quantity = case["quantity"]
                params = {**case, "comment": f"test_{n:04.0f}"}
                response = self.client.post("send", params = params)
                self.assertEqual(response.status_code, 200)
                order_1, order_2 = response.json(), API7._DB.iloc[-1]
                self.assertEqual(order_1["quantity"], abs(quantity))
                self.assertEqual(order_1["side"], numpy.sign(quantity))
                self.assertEqual(order_1["type"], Order.Type.MARKET.value)
                self.assertEqual(order_1["symbol"], case["symbol"])
                order_2: Dict = {"id": order_2.name, **order_2}
                for key in self.KEYS:
                    self.assertEqual(order_1[key], order_2[key])

test = TestAPI
suite = TestLoader().loadTestsFromTestCase(test)
result = TextTestRunner(verbosity = 2).run(suite)
print("Tests run:", result.testsRun, "Failures:", len(result.failures))

test_send (__main__.TestAPI.test_send) ... 

Testing 0 ... Case: {'quantity': 0.1, 'symbol': 'BTCUSDT'}
Testing 1 ... Case: {'quantity': -1.0, 'symbol': 'ETHUSDT'}
Testing 2 ... Case: {'quantity': 10.0, 'symbol': 'BNBUSDT'}
Testing 3 ... Case: {'quantity': -100.0, 'symbol': 'SOLUSDT'}
Testing 4 ... Case: {'quantity': 1000.0, 'symbol': 'UNIUSDT'}
Testing 5 ... Case: {'quantity': -10000.0, 'symbol': 'MATICUSDT'}
Testing 6 ... Case: {'quantity': 100000.0, 'symbol': 'FLOWUSDT'}


ok

----------------------------------------------------------------------
Ran 1 test in 2.540s

OK


Tests run: 1 Failures: 0


## Exercise 8 – Async Routes and External Services

**Goal**: Make async requests from an async route.

**Requirements**:

- Declare an `async def` route that uses `httpx.AsyncClient` to fetch a URL.
- Use `asyncio.gather` inside the route to fetch two URLs concurrently.
- In tests, mock `httpx.AsyncClient.get` to avoid real network calls.

**Hint**: `httpx` has an `AsyncClient` that integrates well with FastAPI's async test helpers.

In [44]:
### No need to solve this exercise. This has been done all along through "Binance.get_price" and "client.get("price")"

## Exercise 9 – Dependency Overrides for Testing

**Goal**: Swap real dependencies for test doubles.

**Requirements**:

- Define a `get_settings()` dependency that reads `os.environ`.
- In a test, use `app.dependency_overrides[get_settings] = lambda: FakeSettings(...)` to inject fixed values.
- Verify the route uses the overridden settings.

**Hint**: `app.dependency_overrides` is a plain dict; clear it in teardown to avoid test pollution.

In [45]:
class API9(API6):

    _APP, _DB = API6._APP, API6._DB
    _DB_EMPTY_TEMP = _DB.copy().iloc[0:0]

class TestAPI(TestCase):

    def setUp(self):
        self.df: DataFrame = API9._DB_EMPTY_TEMP
        self.client = TestClient(app := API9._APP)
        app.dependency_overrides[API9._DB.db] = self.df

    def test_send(self):
        orders = [
            {"quantity": +0.1, "symbol": "BTCUSDT"},
            {"quantity": -1.0, "symbol": "ETHUSDT"},
            {"quantity": +10.0, "symbol": "BNBUSDT"},
            {"quantity": -100.0, "symbol": "SOLUSDT"},
            {"quantity": +1000.0, "symbol": "UNIUSDT"},
            {"quantity": -10000.0, "symbol": "MATICUSDT"},
            {"quantity": +100000.0, "symbol": "FLOWUSDT"},
        ]
        for n, case in enumerate(orders):
            print("Testing", n, "... Case:", case)
            with self.subTest(case = case):
                params = {**case, "comment": f"test_{n:04.0f}"}
                response = self.client.post("send", params = params)
                self.assertEqual(response.status_code, 200)

        self.assertNotEqual(self.df.shape[0], len(orders))

test = TestAPI
suite = TestLoader().loadTestsFromTestCase(test)
result = TextTestRunner(verbosity = 2).run(suite)
print("Tests run:", result.testsRun, "Failures:", len(result.failures))

test_send (__main__.TestAPI.test_send) ... 

Testing 0 ... Case: {'quantity': 0.1, 'symbol': 'BTCUSDT'}
Testing 1 ... Case: {'quantity': -1.0, 'symbol': 'ETHUSDT'}
Testing 2 ... Case: {'quantity': 10.0, 'symbol': 'BNBUSDT'}
Testing 3 ... Case: {'quantity': -100.0, 'symbol': 'SOLUSDT'}
Testing 4 ... Case: {'quantity': 1000.0, 'symbol': 'UNIUSDT'}
Testing 5 ... Case: {'quantity': -10000.0, 'symbol': 'MATICUSDT'}
Testing 6 ... Case: {'quantity': 100000.0, 'symbol': 'FLOWUSDT'}


ok

----------------------------------------------------------------------
Ran 1 test in 2.450s

OK


Tests run: 1 Failures: 0
